In [1]:
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma
from pprint import pprint

#llm = ChatOllama(model="llama3.2:1b") 
llm = ChatOllama(model="llama3.2:3b") # Knowledge cutoff September 2023
#llm = ChatOllama(model="qwen3.5:9b") # Knowledge cutoff October 2024
#llm = ChatOllama(model="qwen3.5:2b")
#llm = ChatOllama(model="gemma4:e2b")
#llm = ChatOllama(model="gemma4:e4b")
#llm = ChatOllama(model="gemma:2b")



#embedding_model = OllamaEmbeddings(model="nomic-embed-text")
embedding_model = OllamaEmbeddings(model="mxbai-embed-large")


events_vector_store = Chroma(
    persist_directory="./chroma_db",
    collection_name="events",
    embedding_function=embedding_model
)

sections_vector_store = Chroma(
    persist_directory="./chroma_db",
    collection_name="sections",
    embedding_function=embedding_model
)

In [2]:
#query = "What are the technical specs of the Nintendo Switch 2 game console?"
#query = "Tell me the news of Nintendo Switch 2"
#query = "What was the market reception of the Nintendo Switch 2 game console?"
#query = "Tell me about Nintendo gaming consoles"
#query = "Compare the features of Nintendo Switch 2 to the original Nintendo Switch"

#query = "What are achievements of  Prema Racing?"
#query = "Tell me about India Maldieve relations in 2025"
#query = "Tell me about the announcements by Apple from 2024 to 2025"
#query = "List the developments of OpenAI in 2025"
#query = "Tell me the major highlights of the Australian Open 2025"
#query = "Tell me about floods in North America in 2025"
#query = "What were the major events of Airbus in 2025" # Limited to dataset
#query = "Tell me about demographic changes in Japan in 2025"
#query = "How did countries adress climate change in 2025?"
#query = "Tell me which wildlife conservation efforts were undertaken in 2025"
#query = "Why were ICE agents in controversies doing in 2025?"
#query = "What were the advances in medicine in 2024 and 2025?"
#query = "How was pollution in Delhi in 2024 and 2025?"
#query = "Tell me about the recent advances in AI and LLMs. Give me their dates too."
query = "What happened in the world of cricket in 2024 to 2025?"

print(f"Query: {query}\n")

print("Generating without using RAG methods...\n")

raw_prompt = ChatPromptTemplate.from_template( 
    """
    Answer this query given all the knowledge you have:
    {query} 
    """
)

chain = raw_prompt | llm | StrOutputParser()
raw_output = chain.invoke({
    "query": query
})

print("Raw Output:\n\n")
print(raw_output)

print("\n\n\nRunning RAG pipeline...")

keywords_prompt = ChatPromptTemplate.from_template( 
    """
    Give me the top 3 very strong aditional single-word keywords I can use in my search to find the answer to this query:
    Query: {query}
    Just output the comma separated keywords and nothing else. I am not asking for future events. Only keywords.
    The keywords should not include what is already in the query except if strongly required by the query.
    Dont provide keywords already present in the query.
    Dont say anything other than the keywords. Don't output anything else and don't complain.
    """
)

chain = keywords_prompt | llm | StrOutputParser()
keywords_output = chain.invoke({
    "query": query
})

print("Keywords: " + keywords_output)

query_vector = embedding_model.embed_query(query + "\nKeywords: " + keywords_output)

top_events = events_vector_store.similarity_search_by_vector(
    query_vector, 
    k=5
)

event_index = {}
section_index = {}

#pprint(top_events)

formatted_events = ""

for event in top_events:
    #print(event)
    event_index[event.id] = event
    # don't use the question and reinsert the date
    event.page_content = event.page_content.split('?\n\n', 1)[1]  
    event.page_content = f"{event.metadata['day']} {event.metadata['month']}, {event.metadata['year']}: {event.page_content}"
    formatted_events += f"\n\n\nEvent {event.id}: {event.page_content}"
    top_sections = sections_vector_store.similarity_search_by_vector(
        query_vector,
        k=5,
        ids=event.metadata["sections"]
    )
    for section in top_sections:
        formatted_events += f"\n\t{event.id}.{section.id}: {section.metadata["title"]}"
        section_index[section.id]=section
    

select_events_prompt = ChatPromptTemplate.from_template( 
    """
    I was this query by the user: {query}

    These events and their sections could be relevant but some of them are not relevant:
        {formatted_events}

    Filter them and select only the events and sections most relevant to the user's query using a maximum of 5 events and 20 sections. Skip events not directly relevant to the user's query.

    If an event is not answering the query, just ignore it and don't include it in the output.
    
    Give the output in a JSON format strictly like this example: {{"e115": ["s899", "s655", ...], "e782": ["s311", ...], ...}}
    The JSON schema is {{"<eventId>": [""<sectionId>"", ""<sectionId>"", ...], ""<eventId>"": [""<sectionId>"", ""<sectionId>"", ...], ...}}
    Only use the correct event IDs in keys and nothing else.
    Just output one object in the JSON and nothing else! 
    """
)

filled_select_events_prompt = select_events_prompt.format(
    query = query,
    formatted_events = formatted_events
)

# print("\nPrompt to select events:")
# print(filled_select_events_prompt)

print(f"\nSelect events prompt length: {len(filled_select_events_prompt)}\n\n")


chain = select_events_prompt | llm | StrOutputParser()
json_str = chain.invoke({
    "query": query,
    "formatted_events": formatted_events
})

clean_json = json_str[json_str.find("{"):json_str.rfind("}") + 1]

print(f"JSON: {clean_json}")

import json

retrieved_content = ""

try:
    data = json.loads(clean_json)
    for ei, (event_id, sections) in enumerate(data.items(), 1):
        event = event_index.get(event_id)
        if event is None: 
            continue
            
        retrieved_content += f"{event.page_content} "
        for ej, section_id in enumerate(sections, 1):
            section_id = section_id.split(".", 1)[-1] # sometimes the format returned is e12313.s2343
            section = section_index.get(section_id)
            if section is None:
                continue
            retrieved_content += f"{section.metadata['summary']} "
        retrieved_content += "\n\n"
    
    print("\nRetrieved content to use: \n")
    print(retrieved_content)

except Exception:
    data = None
    retrieved_content = "No articles found"

# Add today's date to the prompt so that LLM does not infer events in the future
from datetime import datetime
today = datetime.now().strftime("%Y-%m-%d")

final_prompt = ChatPromptTemplate.from_template( 
    """
    Todays date: {today}
    
    I was asked this query by the user: {query}

    Answer the user's query using only relevant parts of the below events:
    {retrieved_content}
    Ignore events and points that are not directly relevant to the query and don't mention them.
    Add factual information from your own knowledge-base to enhance the provided events but do not speculate.
    If events provided are not answering the query fully, use only true facts your own knowledge-base to add to the answer. 
    Before closing, provide an overall summary.
    Do not infer or assume specifications or dates.
    Do not complain in any way about the events provided to you.

    """
)

filled_final_prompt = final_prompt.format(
    query =  query,
    retrieved_content = retrieved_content,
    today = today
)

# print("\nFinal prompt:")
# print(filled_final_prompt)

print(f"\nFinal prompt length: {len(filled_final_prompt)}\n\n")

chain = final_prompt | llm | StrOutputParser()

output = chain.invoke({
    "query": query,
    "retrieved_content": retrieved_content,
    "today": today
})

print("\n\nFinal output:\n")
print(output)

Query: What happened in the world of cricket in 2024 to 2025?

Generating without using RAG methods...

Raw Output:


I don't have any information on future events, including the world of cricket from 2024 to 2025. My training data only goes up until December 2023, and I do not have real-time updates or access to current or future events.

However, I can suggest some possible sources where you may be able to find information on upcoming cricket events:

1. Official cricket websites: Websites like ESPN Cricinfo, Cricket Australia, the International Cricket Council (ICC), and other national cricket boards often provide news, schedules, and updates on upcoming matches.
2. News articles: Online news publications, such as The Guardian, The Times of India, and other sports-focused media outlets, usually publish news on upcoming cricket events.
3. Social media: Follow official cricket accounts, teams, players, and media outlets on social media platforms like Twitter, Instagram, and Facebook t